In [2]:
"""
=============================================================================
NCAA March Mania 2026 — shaurya_final_5.py
=============================================================================
✅ Runs locally on Jupyter (auto-detects Downloads folder)
✅ Also runs on Kaggle (auto-detects Kaggle path)
✅ 2026 regular season stats for prediction features
✅ 2026 tournament seeds (all 68 teams)
✅ 2026 ELO computed from 2026 regular season games
✅ 2026 Massey ordinals (pre-tournament)
✅ 2026 recent form (last 10 games)
✅ Train: 2010-2025 | Men + Women pipelines
✅ First Four results locked in
✅ Output: shaurya_final_5.csv
=============================================================================
"""

import warnings; warnings.filterwarnings("ignore")
import os, re, multiprocessing
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss
import lightgbm as lgb
import xgboost as xgb

# ─────────────────────────────────────────────────────────────
# AUTO-DETECT PATHS  (Jupyter local OR Kaggle)
# ─────────────────────────────────────────────────────────────
def find_data_dir():
    candidates = [
        Path("/Users/shaurya/Downloads"),
        Path.home() / "Downloads",
        Path("/kaggle/input/march-machine-learning-mania-2026"),
    ]
    for p in candidates:
        if p.exists() and (p / "SampleSubmissionStage2.csv").exists():
            return p
    raise FileNotFoundError(
        "Could not find data folder.\n"
        "Please manually set DATA_DIR = Path('/your/path/to/data') below."
    )

DATA_DIR = find_data_dir()
OUT_DIR  = Path("/kaggle/working") if Path("/kaggle/working").exists() else DATA_DIR

print(f"Data dir  : {DATA_DIR}")
print(f"Output dir: {OUT_DIR}")

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
N_JOBS = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"]      = str(N_JOBS)
os.environ["MKL_NUM_THREADS"]      = str(N_JOBS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_JOBS)
print(f"CPU cores : {N_JOBS}")

TRAIN_SEASONS = list(range(2010, 2026))
PRED_SEASON   = 2026
ELO_K         = 20
ELO_BASE      = 1500
ELO_HOME_ADV  = 100
SEED          = 42
np.random.seed(SEED)

STAT_COLS = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
             "OR","DR","Ast","TO","Stl","Blk","PF"]

# ─────────────────────────────────────────────────────────────
# MODEL PARAMS — MEN'S
# ─────────────────────────────────────────────────────────────
MEN_LGBM = dict(
    num_leaves=170, max_depth=12, learning_rate=0.005,
    n_estimators=1508, min_child_samples=5,
    subsample=0.5, colsample_bytree=0.5,
    reg_alpha=0.0, reg_lambda=0.0,
    objective="binary", metric="binary_logloss",
    random_state=SEED, n_jobs=N_JOBS, verbose=-1,
)
MEN_XGB = dict(
    n_estimators=1838, max_depth=10, learning_rate=0.0255778,
    min_child_weight=2.84511, subsample=0.848826,
    colsample_bytree=0.562814, gamma=1.59748,
    reg_alpha=4.86392, reg_lambda=1.04622,
    early_stopping_rounds=50,
    objective="binary:logistic", eval_metric="logloss",
    random_state=SEED, n_jobs=N_JOBS, verbosity=0, tree_method="hist",
)

# ─────────────────────────────────────────────────────────────
# MODEL PARAMS — WOMEN'S
# ─────────────────────────────────────────────────────────────
WOMEN_LGBM = dict(
    num_leaves=164, max_depth=8, learning_rate=0.0187029,
    n_estimators=1494, min_child_samples=22,
    subsample=0.532526, colsample_bytree=0.974443,
    reg_alpha=4.82816, reg_lambda=4.04199,
    objective="binary", metric="binary_logloss",
    random_state=SEED, n_jobs=N_JOBS, verbose=-1,
)
WOMEN_XGB = dict(
    n_estimators=2098, max_depth=7, learning_rate=0.0559337,
    min_child_weight=18.9736, subsample=0.842968,
    colsample_bytree=0.895046, gamma=0.171643,
    reg_alpha=1.13418, reg_lambda=2.44057,
    early_stopping_rounds=50,
    objective="binary:logistic", eval_metric="logloss",
    random_state=SEED, n_jobs=N_JOBS, verbosity=0, tree_method="hist",
)

# ─────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────

def parse_seed_num(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16


def explode_games(games_df):
    """Convert W/L rows into two team-perspective rows per game."""
    if len(games_df) == 0:
        cols = (["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst","Win"]
                + STAT_COLS + [f"Opp{s}" for s in STAT_COLS])
        return pd.DataFrame(columns=cols)

    w = games_df.copy()
    w["TeamID"] = w["WTeamID"]; w["OppID"] = w["LTeamID"]
    w["PtsFor"] = w["WScore"];  w["PtsAgainst"] = w["LScore"]
    w["Win"] = 1
    for s in STAT_COLS:
        w[s] = w[f"W{s}"]; w[f"Opp{s}"] = w[f"L{s}"]

    l = games_df.copy()
    l["TeamID"] = l["LTeamID"]; l["OppID"] = l["WTeamID"]
    l["PtsFor"] = l["LScore"];  l["PtsAgainst"] = l["WScore"]
    l["Win"] = 0
    for s in STAT_COLS:
        l[s] = l[f"L{s}"]; l[f"Opp{s}"] = l[f"W{s}"]

    keep = (["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst","Win"]
            + STAT_COLS + [f"Opp{s}" for s in STAT_COLS])
    return pd.concat([w[keep], l[keep]], ignore_index=True)


def compute_elo_seasons(reg_df, tourney_df, seasons):
    """ELO across training seasons. Returns (elo_dict, end_of_last_season_elo)."""
    cols = ["Season","DayNum","WTeamID","LTeamID","WLoc","WScore","LScore"]
    all_games = (pd.concat([reg_df[cols], tourney_df[cols]], ignore_index=True)
                   .sort_values(["Season","DayNum"]).reset_index(drop=True))
    all_games = all_games[all_games["Season"].isin(seasons)]

    elo = {}
    elo_dict = {}

    for season in sorted(seasons):
        elo = {t: ELO_BASE * 0.25 + v * 0.75 for t, v in elo.items()}
        for _, r in all_games[all_games["Season"] == season].iterrows():
            w, l   = int(r["WTeamID"]), int(r["LTeamID"])
            rw, rl = elo.get(w, ELO_BASE), elo.get(l, ELO_BASE)
            loc    = r["WLoc"]
            rw_adj = rw + ELO_HOME_ADV if loc=="H" else rw - ELO_HOME_ADV if loc=="A" else rw
            mov    = min(abs(int(r["WScore"]) - int(r["LScore"])), 20)
            mov_m  = np.log1p(mov) * (2.2 / ((rw_adj - rl) * 0.001 + 2.2))
            exp_w  = 1 / (1 + 10 ** ((rl - rw_adj) / 400))
            delta  = ELO_K * mov_m * (1 - exp_w)
            elo[w] = rw + delta
            elo[l] = rl - delta
        for tid, rat in elo.items():
            elo_dict[(season, tid)] = rat

    return elo_dict, dict(elo)


def compute_elo_2026(reg_2026_df, elo_end_2025):
    """Extend ELO through 2026 regular season from end-of-2025 base."""
    elo = {t: ELO_BASE * 0.25 + v * 0.75 for t, v in elo_end_2025.items()}
    for _, r in reg_2026_df.sort_values("DayNum").iterrows():
        w, l   = int(r["WTeamID"]), int(r["LTeamID"])
        rw, rl = elo.get(w, ELO_BASE), elo.get(l, ELO_BASE)
        loc    = r["WLoc"]
        rw_adj = rw + ELO_HOME_ADV if loc=="H" else rw - ELO_HOME_ADV if loc=="A" else rw
        mov    = min(abs(int(r["WScore"]) - int(r["LScore"])), 20)
        mov_m  = np.log1p(mov) * (2.2 / ((rw_adj - rl) * 0.001 + 2.2))
        exp_w  = 1 / (1 + 10 ** ((rl - rw_adj) / 400))
        delta  = ELO_K * mov_m * (1 - exp_w)
        elo[w] = rw + delta
        elo[l] = rl - delta
    return elo


def build_team_stats(reg_df, tourney_df):
    """Advanced stats per (Season, TeamID)."""
    eps = 1e-6
    tg  = explode_games(pd.concat([reg_df, tourney_df], ignore_index=True))
    if len(tg) == 0:
        return pd.DataFrame()

    agg = tg.groupby(["Season","TeamID"]).agg(
        GP=("Win","count"), Wins=("Win","sum"),
        PtsFor=("PtsFor","mean"), PtsAgainst=("PtsAgainst","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OR=("OR","mean"),   DR=("DR","mean"),
        Ast=("Ast","mean"), TO=("TO","mean"),
        Stl=("Stl","mean"), Blk=("Blk","mean"), PF=("PF","mean"),
        OppFGM=("OppFGM","mean"),   OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
        OppOR=("OppOR","mean"),     OppDR=("OppDR","mean"),
        OppTO=("OppTO","mean"),
    ).reset_index()

    agg["WinPct"]         = agg["Wins"] / (agg["GP"] + eps)
    agg["FGPct"]          = agg["FGM"]  / (agg["FGA"]  + eps)
    agg["FG3Pct"]         = agg["FGM3"] / (agg["FGA3"] + eps)
    agg["FTPct"]          = agg["FTM"]  / (agg["FTA"]  + eps)
    agg["OppFGPct"]       = agg["OppFGM"]  / (agg["OppFGA"]  + eps)
    agg["eFGPct"]         = (agg["FGM"]    + 0.5*agg["FGM3"])    / (agg["FGA"]    + eps)
    agg["OppEFGPct"]      = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / (agg["OppFGA"] + eps)
    poss                  = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    agg["TOVRate"]        = agg["TO"]          / (poss + eps)
    agg["ORBRate"]        = agg["OR"]          / (agg["OR"]  + agg["OppDR"] + eps)
    agg["FTRate"]         = agg["FTM"]         / (agg["FGA"] + eps)
    agg["DRBRate"]        = agg["DR"]          / (agg["DR"]  + agg["OppOR"] + eps)
    agg["Pace"]           = poss               / (agg["GP"]  + eps)
    agg["NetRtg"]         = agg["PtsFor"]      - agg["PtsAgainst"]
    agg["OffRtg"]         = agg["PtsFor"]      / (poss + eps) * 100
    agg["DefRtg"]         = agg["PtsAgainst"]  / (poss + eps) * 100
    agg["NetRtg2"]        = agg["OffRtg"]      - agg["DefRtg"]
    agg["AstTO"]          = agg["Ast"]         / (agg["TO"]     + eps)
    agg["StlBlk"]         = (agg["Stl"] + agg["Blk"]) / (agg["OppFGA"] + eps)
    agg["ThreePtRate"]    = agg["FGA3"]        / (agg["FGA"]    + eps)
    agg["OppThreePtRate"] = agg["OppFGA3"]     / (agg["OppFGA"] + eps)
    agg["TOMargin"]       = agg["OppTO"]       - agg["TO"]
    agg["RebMargin"]      = agg["OR"]          - agg["OppOR"]
    return agg


def build_recent_form(reg_df, n=10):
    """Last n regular season games per team."""
    tg  = explode_games(reg_df).sort_values(["Season","TeamID","DayNum"])
    rec = tg.groupby(["Season","TeamID"]).tail(n)
    rf  = rec.groupby(["Season","TeamID"]).agg(
        RecentWinPct    =("Win","mean"),
        RecentPtsFor    =("PtsFor","mean"),
        RecentPtsAgainst=("PtsAgainst","mean"),
    ).reset_index()
    rf["RecentNetPts"] = rf["RecentPtsFor"] - rf["RecentPtsAgainst"]
    return rf


def build_massey(seasons, day_cutoff=133):
    """Massey ordinals averaged across all systems."""
    fp = DATA_DIR / "MMasseyOrdinals.csv"
    if not fp.exists():
        print("  ℹ️  MMasseyOrdinals.csv not found — skipping")
        return None
    df = pd.read_csv(fp)
    df = df[(df["Season"].isin(seasons)) & (df["RankingDayNum"] <= day_cutoff)]
    if len(df) == 0:
        return None
    result = (df.groupby(["Season","TeamID"])["OrdinalRank"]
                .mean().reset_index()
                .rename(columns={"OrdinalRank": "AvgRank"}))
    for sys in ["SAG","POM","WOL","MOR","DOK","REW","NET"]:
        sub = df[df["SystemName"] == sys]
        if len(sub) == 0:
            continue
        s = (sub.groupby(["Season","TeamID"])["OrdinalRank"]
                .mean().reset_index()
                .rename(columns={"OrdinalRank": f"Rank_{sys}"}))
        result = result.merge(s, on=["Season","TeamID"], how="left")
    return result


def attach(df, ref, side, team_col, season_join=True):
    """
    Merge a reference table onto df for one side of a matchup.
    side      : 'A' or 'B'
    team_col  : column in df and ref that holds the team ID
    season_join: whether to also join on Season
    """
    ref    = ref.copy()
    f_cols = [c for c in ref.columns if c not in [team_col, "Season"]]
    ref    = ref.rename(columns={c: f"{side}_{c}" for c in f_cols})
    keys   = ([team_col, "Season"] if season_join and "Season" in ref.columns
              else [team_col])
    return df.merge(ref, on=keys, how="left")


def make_matchup_features(games_df, team_stats, recent, seed_feat,
                           massey_feat, elo_dict):
    """Build training matchup matrix. TeamA = lower ID, Label = 1 if TeamA won."""
    df          = games_df.copy()
    df["TeamA"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["TeamB"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Label"] = (df["WTeamID"] == df["TeamA"]).astype(int)

    rf = recent[["Season","TeamID","RecentWinPct","RecentNetPts",
                 "RecentPtsFor","RecentPtsAgainst"]].copy()
    sd = seed_feat[["Season","TeamID","SeedNum"]].copy()

    # Rename TeamID to TeamA / TeamB before attaching
    df = attach(df, team_stats.rename(columns={"TeamID":"TeamA"}), "A", "TeamA")
    df = attach(df, team_stats.rename(columns={"TeamID":"TeamB"}), "B", "TeamB")
    df = attach(df, rf.rename(columns={"TeamID":"TeamA"}),         "A", "TeamA")
    df = attach(df, rf.rename(columns={"TeamID":"TeamB"}),         "B", "TeamB")
    df = attach(df, sd.rename(columns={"TeamID":"TeamA"}),         "A", "TeamA")
    df = attach(df, sd.rename(columns={"TeamID":"TeamB"}),         "B", "TeamB")

    if massey_feat is not None:
        mc = [c for c in massey_feat.columns if c not in ["Season","TeamID"]]
        mf = massey_feat[["Season","TeamID"] + mc].copy()
        df = attach(df, mf.rename(columns={"TeamID":"TeamA"}), "A", "TeamA")
        df = attach(df, mf.rename(columns={"TeamID":"TeamB"}), "B", "TeamB")

    # Prior end-of-season ELO
    df["A_ELO"] = df.apply(
        lambda r: elo_dict.get((int(r["Season"])-1, int(r["TeamA"])), ELO_BASE), axis=1)
    df["B_ELO"] = df.apply(
        lambda r: elo_dict.get((int(r["Season"])-1, int(r["TeamB"])), ELO_BASE), axis=1)

    feature_cols = []
    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]
            feature_cols.extend([f"diff_{base}", ca, cb])

    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    feature_cols      = list(dict.fromkeys(feature_cols + ["ELO_WinProb"]))
    return df, feature_cols


def build_pred_features_2026(pairs_df, stats_2026, recent_2026,
                              seed_2026, massey_2026, elo_2026_final,
                              feature_cols):
    """
    Build 2026 prediction features.
    All tables are single-season (Season col dropped before merging).
    """
    df = pairs_df[["TeamA","TeamB"]].copy().reset_index(drop=True)

    def drop_season(t):
        return t.drop(columns=["Season"], errors="ignore").copy()

    s26  = drop_season(stats_2026)
    rf26 = drop_season(recent_2026)[["TeamID","RecentWinPct","RecentNetPts",
                                      "RecentPtsFor","RecentPtsAgainst"]]
    sd26 = drop_season(seed_2026)[["TeamID","SeedNum"]]

    df = attach(df, s26.rename(columns={"TeamID":"TeamA"}),  "A", "TeamA",  season_join=False)
    df = attach(df, s26.rename(columns={"TeamID":"TeamB"}),  "B", "TeamB",  season_join=False)
    df = attach(df, rf26.rename(columns={"TeamID":"TeamA"}), "A", "TeamA",  season_join=False)
    df = attach(df, rf26.rename(columns={"TeamID":"TeamB"}), "B", "TeamB",  season_join=False)
    df = attach(df, sd26.rename(columns={"TeamID":"TeamA"}), "A", "TeamA",  season_join=False)
    df = attach(df, sd26.rename(columns={"TeamID":"TeamB"}), "B", "TeamB",  season_join=False)

    if massey_2026 is not None and len(massey_2026) > 0:
        m26 = drop_season(massey_2026)
        mc  = [c for c in m26.columns if c != "TeamID"]
        df  = attach(df, m26[["TeamID"]+mc].rename(columns={"TeamID":"TeamA"}),
                     "A", "TeamA", season_join=False)
        df  = attach(df, m26[["TeamID"]+mc].rename(columns={"TeamID":"TeamB"}),
                     "B", "TeamB", season_join=False)

    # 2026 ELO
    df["A_ELO"] = df["TeamA"].apply(lambda t: elo_2026_final.get(int(t), ELO_BASE))
    df["B_ELO"] = df["TeamB"].apply(lambda t: elo_2026_final.get(int(t), ELO_BASE))

    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]

    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))

    X = df.reindex(columns=feature_cols)
    n_nan_cols = (X.isna().sum() > 0).sum()
    if n_nan_cols > 0:
        print(f"    ℹ️  {n_nan_cols} columns had NaNs (filled 0) — normal for unseeded teams")
    return X.fillna(0)


# ─────────────────────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────────────────────
def run_pipeline(reg_full, tourney_full, seeds_full, use_massey,
                 lgbm_params, xgb_params, pred_pairs, label):

    print(f"\n{'='*65}")
    print(f"  {label}")
    print(f"{'='*65}")

    reg_train     = reg_full[reg_full["Season"].isin(TRAIN_SEASONS)].copy()
    tourney_train = tourney_full[tourney_full["Season"].isin(TRAIN_SEASONS)].copy()
    reg_2026      = reg_full[reg_full["Season"] == PRED_SEASON].copy()

    print(f"  Train reg    : {len(reg_train):,} games "
          f"({reg_train.Season.min()}-{reg_train.Season.max()})")
    print(f"  Train tourney: {len(tourney_train):,} games")
    print(f"  2026 reg     : {len(reg_2026):,} games <- prediction basis")

    # ELO
    print(f"\n  [ELO] Training seasons 2010-2025 ...")
    elo_dict, elo_end_2025 = compute_elo_seasons(reg_train, tourney_train, TRAIN_SEASONS)
    print(f"        {len(elo_dict):,} (season,team) entries")

    print(f"  [ELO] Extending through 2026 regular season ...")
    elo_2026 = compute_elo_2026(reg_2026, elo_end_2025)
    print(f"        {len(elo_2026):,} teams rated for 2026")

    # Training stats
    print(f"\n  [Stats] Building 2010-2025 training stats ...")
    team_stats  = build_team_stats(reg_train, tourney_train)
    recent      = build_recent_form(reg_train)
    seed_feat   = seeds_full[seeds_full["Season"].isin(TRAIN_SEASONS)].copy()
    seed_feat["SeedNum"] = seed_feat["Seed"].apply(parse_seed_num)
    seed_feat   = seed_feat[["Season","TeamID","SeedNum"]]
    massey_train = build_massey(TRAIN_SEASONS) if use_massey else None
    print(f"        {len(team_stats):,} team-season rows")
    print(f"        Massey: {'yes' if massey_train is not None else 'N/A'}")

    # Training matchup matrix
    print(f"\n  [Features] Building training matchup matrix ...")
    all_train = pd.concat([reg_train, tourney_train], ignore_index=True)
    train_df, feature_cols = make_matchup_features(
        all_train, team_stats, recent, seed_feat, massey_train, elo_dict)
    X_tr = train_df[feature_cols].fillna(0)
    y_tr = train_df["Label"]
    print(f"        {len(X_tr):,} rows x {len(feature_cols)} features")

    # Train LightGBM
    print(f"\n  [Train] LightGBM ...")
    lgbm_m = lgb.LGBMClassifier(**lgbm_params)
    lgbm_m.fit(X_tr, y_tr)
    print(f"         done")

    # Train XGBoost (needs val split for early stopping)
    split  = int(len(X_tr) * 0.9)
    X_trx, y_trx   = X_tr.iloc[:split],  y_tr.iloc[:split]
    X_valx, y_valx = X_tr.iloc[split:],  y_tr.iloc[split:]

    print(f"  [Train] XGBoost ...")
    xgb_m = xgb.XGBClassifier(**xgb_params)
    xgb_m.fit(X_trx, y_trx, eval_set=[(X_valx, y_valx)], verbose=False)
    print(f"         done")

    # Tune blend weight
    p_lv = lgbm_m.predict_proba(X_valx)[:,1]
    p_xv = xgb_m.predict_proba(X_valx)[:,1]
    best_w, best_ll = 0.5, 99.0
    for w in np.arange(0.05, 0.96, 0.05):
        ll = log_loss(y_valx, w*p_lv + (1-w)*p_xv)
        if ll < best_ll:
            best_ll, best_w = ll, w
    print(f"  [Blend] LGB:{best_w:.2f}  XGB:{1-best_w:.2f}  "
          f"val_logloss={best_ll:.5f}")

    # 2026 prediction features  <-- THE KEY FIX
    print(f"\n  [2026] Building 2026 prediction features ...")
    empty_df    = pd.DataFrame(columns=reg_2026.columns)
    stats_2026  = build_team_stats(reg_2026, empty_df)
    stats_2026  = stats_2026[stats_2026["Season"] == PRED_SEASON].copy()
    recent_2026 = build_recent_form(reg_2026)
    recent_2026 = recent_2026[recent_2026["Season"] == PRED_SEASON].copy()
    seed_2026   = seeds_full[seeds_full["Season"] == PRED_SEASON].copy()
    seed_2026["SeedNum"] = seed_2026["Seed"].apply(parse_seed_num)
    seed_2026   = seed_2026[["Season","TeamID","SeedNum"]]
    massey_2026 = None
    if use_massey:
        massey_2026 = build_massey([PRED_SEASON])
        if massey_2026 is not None:
            massey_2026 = massey_2026[massey_2026["Season"] == PRED_SEASON].copy()

    print(f"        stats : {len(stats_2026):,} teams")
    print(f"        recent: {len(recent_2026):,} teams")
    print(f"        seeds : {len(seed_2026):,} teams")
    print(f"        massey: {len(massey_2026) if massey_2026 is not None else 'N/A'}")

    X_pred = build_pred_features_2026(
        pred_pairs, stats_2026, recent_2026,
        seed_2026, massey_2026, elo_2026, feature_cols)
    print(f"        {len(X_pred):,} prediction rows")

    # Predict
    print(f"\n  [Predict] Running ensemble ...")
    p_lgbm = lgbm_m.predict_proba(X_pred)[:,1]
    p_xgb  = xgb_m.predict_proba(X_pred)[:,1]
    p_ens  = np.clip(best_w*p_lgbm + (1-best_w)*p_xgb, 0.025, 0.975)
    print(f"         mean={p_ens.mean():.4f}  "
          f"min={p_ens.min():.4f}  max={p_ens.max():.4f}")
    return p_ens


# ─────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  LOADING DATA")
print("="*65)

stage2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")
parts           = stage2["ID"].str.split("_", expand=True)
stage2["Season"] = parts[0].astype(int)
stage2["TeamA"]  = parts[1].astype(int)
stage2["TeamB"]  = parts[2].astype(int)

men_mask    = stage2["TeamA"] < 3000
women_mask  = stage2["TeamA"] >= 3000
men_pairs   = stage2[men_mask][["TeamA","TeamB"]].reset_index(drop=True)
women_pairs = stage2[women_mask][["TeamA","TeamB"]].reset_index(drop=True)

print(f"  Submission total: {len(stage2):,}")
print(f"  Men    (1xxx)   : {len(men_pairs):,}")
print(f"  Women  (3xxx)   : {len(women_pairs):,}")

print("\n  Men's files ...")
m_reg_full     = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney_full = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
m_seeds_full   = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
print(f"  M reg {m_reg_full.Season.min()}-{m_reg_full.Season.max()} | "
      f"2026 games: {len(m_reg_full[m_reg_full.Season==2026]):,}")
print(f"  M seeds 2026: {len(m_seeds_full[m_seeds_full.Season==2026])} teams")

print("\n  Women's files ...")
w_reg_full     = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney_full = pd.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")
w_seeds_full   = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
print(f"  W reg {w_reg_full.Season.min()}-{w_reg_full.Season.max()} | "
      f"2026 games: {len(w_reg_full[w_reg_full.Season==2026]):,}")
print(f"  W seeds 2026: {len(w_seeds_full[w_seeds_full.Season==2026])} teams")

# ─────────────────────────────────────────────────────────────
# RUN PIPELINES
# ─────────────────────────────────────────────────────────────
men_preds = run_pipeline(
    m_reg_full, m_tourney_full, m_seeds_full,
    use_massey=True,
    lgbm_params=MEN_LGBM, xgb_params=MEN_XGB,
    pred_pairs=men_pairs,
    label="MENS PIPELINE"
)

women_preds = run_pipeline(
    w_reg_full, w_tourney_full, w_seeds_full,
    use_massey=False,
    lgbm_params=WOMEN_LGBM, xgb_params=WOMEN_XGB,
    pred_pairs=women_pairs,
    label="WOMENS PIPELINE"
)

# ─────────────────────────────────────────────────────────────
# ASSEMBLE + SAVE
# ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  ASSEMBLING SUBMISSION")
print("="*65)

men_out         = stage2[men_mask][["ID"]].copy().reset_index(drop=True)
men_out["Pred"] = men_preds.round(6)

women_out         = stage2[women_mask][["ID"]].copy().reset_index(drop=True)
women_out["Pred"] = women_preds.round(6)

submission = (stage2[["ID"]]
              .merge(pd.concat([men_out, women_out], ignore_index=True),
                     on="ID", how="left"))
submission["Pred"] = submission["Pred"].clip(0.025, 0.975).round(6)

# Lock confirmed First Four results
# ID format: 2026_loTeamID_hiTeamID | Pred = P(loTeamID wins)
known_results = {
    "2026_1224_1420": 1.0,   # Howard beat UMBC       86-83
    "2026_1301_1400": 0.0,   # Texas  beat NC State   68-66  (1301=NC State lost)
    "2026_1250_1341": 0.0,   # Prairie View beat Lehigh 67-55 (1250=Lehigh lost)
    "2026_1275_1374": 1.0,   # Miami OH beat SMU       89-79  (1275=Miami OH won)
}
id_idx = {row["ID"]: i for i, row in submission.iterrows()}
for mid, val in known_results.items():
    if mid in id_idx:
        submission.at[id_idx[mid], "Pred"] = val
        print(f"  Locked: {mid} -> {val}")

# Save
out_path = OUT_DIR / "shaurya_final_5.csv"
submission[["ID","Pred"]].to_csv(out_path, index=False)

# Sanity checks
assert len(submission) == len(stage2),               "Row count mismatch"
assert submission["ID"].nunique() == len(submission), "Duplicate IDs"
assert submission["Pred"].between(0,1).all(),         "Pred outside [0,1]"
assert submission["Pred"].isna().sum() == 0,          "NaN predictions"

print(f"\n{'='*65}")
print(f"  DONE")
print(f"  File    : {out_path}")
print(f"  Rows    : {len(submission):,}  (expected 132,133)")
print(f"  NaNs    : {submission['Pred'].isna().sum()}")
print(f"  Min pred: {submission['Pred'].min():.4f}")
print(f"  Max pred: {submission['Pred'].max():.4f}")
print(f"  Mean    : {submission['Pred'].mean():.4f}")
print(f"{'='*65}")
print(f"\n  shaurya_final_5.csv is ready to submit on Kaggle!")
print(submission.head(5).to_string(index=False))

Data dir  : /Users/shaurya/Downloads
Output dir: /Users/shaurya/Downloads
CPU cores : 8

  LOADING DATA
  Submission total: 132,133
  Men    (1xxx)   : 66,430
  Women  (3xxx)   : 65,703

  Men's files ...
  M reg 2003-2026 | 2026 games: 5,647
  M seeds 2026: 68 teams

  Women's files ...
  W reg 2010-2026 | 2026 games: 5,479
  W seeds 2026: 68 teams

  MENS PIPELINE
  Train reg    : 84,808 games (2010-2025)
  Train tourney: 1,001 games
  2026 reg     : 5,647 games <- prediction basis

  [ELO] Training seasons 2010-2025 ...
        5,690 (season,team) entries
  [ELO] Extending through 2026 regular season ...
        370 teams rated for 2026

  [Stats] Building 2010-2025 training stats ...
        5,639 team-season rows
        Massey: yes

  [Features] Building training matchup matrix ...
        85,809 rows x 181 features

  [Train] LightGBM ...
         done
  [Train] XGBoost ...
         done
  [Blend] LGB:0.95  XGB:0.05  val_logloss=0.40212

  [2026] Building 2026 prediction feature

In [3]:
"""
=============================================================================
NCAA March Mania 2026 — shaurya_final_6.py
=============================================================================
✅ Runs locally on Jupyter (auto-detects Downloads folder)
✅ Also runs on Kaggle (auto-detects Kaggle path)
✅ 2026 regular season stats for prediction features
✅ 2026 tournament seeds (all 68 teams)
✅ 2026 ELO computed from 2026 regular season games
✅ 2026 Massey ordinals (pre-tournament)
✅ 2026 recent form (last 10 games)
✅ Train: 2010-2025 | Men + Women pipelines
✅ First Four results locked in
✅ Conference Tournament features added (Men + Women)
✅ Output: shaurya_final_6.csv
=============================================================================
"""

import warnings; warnings.filterwarnings("ignore")
import os, re, multiprocessing
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss
import lightgbm as lgb
import xgboost as xgb

# ─────────────────────────────────────────────────────────────
# AUTO-DETECT PATHS  (Jupyter local OR Kaggle)
# ─────────────────────────────────────────────────────────────
def find_data_dir():
    candidates = [
        Path("/Users/shaurya/Downloads"),
        Path.home() / "Downloads",
        Path("/kaggle/input/march-machine-learning-mania-2026"),
    ]
    for p in candidates:
        if p.exists() and (p / "SampleSubmissionStage2.csv").exists():
            return p
    raise FileNotFoundError(
        "Could not find data folder.\n"
        "Please manually set DATA_DIR = Path('/your/path/to/data') below."
    )

DATA_DIR = find_data_dir()
OUT_DIR  = Path("/kaggle/working") if Path("/kaggle/working").exists() else DATA_DIR

print(f"Data dir  : {DATA_DIR}")
print(f"Output dir: {OUT_DIR}")

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
N_JOBS = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"]      = str(N_JOBS)
os.environ["MKL_NUM_THREADS"]      = str(N_JOBS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_JOBS)
print(f"CPU cores : {N_JOBS}")

TRAIN_SEASONS = list(range(2010, 2026))
PRED_SEASON   = 2026
ELO_K         = 20
ELO_BASE      = 1500
ELO_HOME_ADV  = 100
SEED          = 42
np.random.seed(SEED)

STAT_COLS = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
             "OR","DR","Ast","TO","Stl","Blk","PF"]

# ─────────────────────────────────────────────────────────────
# MODEL PARAMS — MEN'S
# ─────────────────────────────────────────────────────────────
MEN_LGBM = dict(
    num_leaves=170, max_depth=12, learning_rate=0.005,
    n_estimators=1508, min_child_samples=5,
    subsample=0.5, colsample_bytree=0.5,
    reg_alpha=0.0, reg_lambda=0.0,
    objective="binary", metric="binary_logloss",
    random_state=SEED, n_jobs=N_JOBS, verbose=-1,
)
MEN_XGB = dict(
    n_estimators=1838, max_depth=10, learning_rate=0.0255778,
    min_child_weight=2.84511, subsample=0.848826,
    colsample_bytree=0.562814, gamma=1.59748,
    reg_alpha=4.86392, reg_lambda=1.04622,
    early_stopping_rounds=50,
    objective="binary:logistic", eval_metric="logloss",
    random_state=SEED, n_jobs=N_JOBS, verbosity=0, tree_method="hist",
)

# ─────────────────────────────────────────────────────────────
# MODEL PARAMS — WOMEN'S
# ─────────────────────────────────────────────────────────────
WOMEN_LGBM = dict(
    num_leaves=164, max_depth=8, learning_rate=0.0187029,
    n_estimators=1494, min_child_samples=22,
    subsample=0.532526, colsample_bytree=0.974443,
    reg_alpha=4.82816, reg_lambda=4.04199,
    objective="binary", metric="binary_logloss",
    random_state=SEED, n_jobs=N_JOBS, verbose=-1,
)
WOMEN_XGB = dict(
    n_estimators=2098, max_depth=7, learning_rate=0.0559337,
    min_child_weight=18.9736, subsample=0.842968,
    colsample_bytree=0.895046, gamma=0.171643,
    reg_alpha=1.13418, reg_lambda=2.44057,
    early_stopping_rounds=50,
    objective="binary:logistic", eval_metric="logloss",
    random_state=SEED, n_jobs=N_JOBS, verbosity=0, tree_method="hist",
)

# ─────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────

def parse_seed_num(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16


def explode_games(games_df):
    """Convert W/L rows into two team-perspective rows per game."""
    if len(games_df) == 0:
        cols = (["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst","Win"]
                + STAT_COLS + [f"Opp{s}" for s in STAT_COLS])
        return pd.DataFrame(columns=cols)

    w = games_df.copy()
    w["TeamID"] = w["WTeamID"]; w["OppID"] = w["LTeamID"]
    w["PtsFor"] = w["WScore"];  w["PtsAgainst"] = w["LScore"]
    w["Win"] = 1
    for s in STAT_COLS:
        w[s] = w[f"W{s}"]; w[f"Opp{s}"] = w[f"L{s}"]

    l = games_df.copy()
    l["TeamID"] = l["LTeamID"]; l["OppID"] = l["WTeamID"]
    l["PtsFor"] = l["LScore"];  l["PtsAgainst"] = l["WScore"]
    l["Win"] = 0
    for s in STAT_COLS:
        l[s] = l[f"L{s}"]; l[f"Opp{s}"] = l[f"W{s}"]

    keep = (["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst","Win"]
            + STAT_COLS + [f"Opp{s}" for s in STAT_COLS])
    return pd.concat([w[keep], l[keep]], ignore_index=True)


def compute_elo_seasons(reg_df, tourney_df, seasons):
    """ELO across training seasons. Returns (elo_dict, end_of_last_season_elo)."""
    cols = ["Season","DayNum","WTeamID","LTeamID","WLoc","WScore","LScore"]
    all_games = (pd.concat([reg_df[cols], tourney_df[cols]], ignore_index=True)
                   .sort_values(["Season","DayNum"]).reset_index(drop=True))
    all_games = all_games[all_games["Season"].isin(seasons)]

    elo = {}
    elo_dict = {}

    for season in sorted(seasons):
        elo = {t: ELO_BASE * 0.25 + v * 0.75 for t, v in elo.items()}
        for _, r in all_games[all_games["Season"] == season].iterrows():
            w, l   = int(r["WTeamID"]), int(r["LTeamID"])
            rw, rl = elo.get(w, ELO_BASE), elo.get(l, ELO_BASE)
            loc    = r["WLoc"]
            rw_adj = rw + ELO_HOME_ADV if loc=="H" else rw - ELO_HOME_ADV if loc=="A" else rw
            mov    = min(abs(int(r["WScore"]) - int(r["LScore"])), 20)
            mov_m  = np.log1p(mov) * (2.2 / ((rw_adj - rl) * 0.001 + 2.2))
            exp_w  = 1 / (1 + 10 ** ((rl - rw_adj) / 400))
            delta  = ELO_K * mov_m * (1 - exp_w)
            elo[w] = rw + delta
            elo[l] = rl - delta
        for tid, rat in elo.items():
            elo_dict[(season, tid)] = rat

    return elo_dict, dict(elo)


def compute_elo_2026(reg_2026_df, elo_end_2025):
    """Extend ELO through 2026 regular season from end-of-2025 base."""
    elo = {t: ELO_BASE * 0.25 + v * 0.75 for t, v in elo_end_2025.items()}
    for _, r in reg_2026_df.sort_values("DayNum").iterrows():
        w, l   = int(r["WTeamID"]), int(r["LTeamID"])
        rw, rl = elo.get(w, ELO_BASE), elo.get(l, ELO_BASE)
        loc    = r["WLoc"]
        rw_adj = rw + ELO_HOME_ADV if loc=="H" else rw - ELO_HOME_ADV if loc=="A" else rw
        mov    = min(abs(int(r["WScore"]) - int(r["LScore"])), 20)
        mov_m  = np.log1p(mov) * (2.2 / ((rw_adj - rl) * 0.001 + 2.2))
        exp_w  = 1 / (1 + 10 ** ((rl - rw_adj) / 400))
        delta  = ELO_K * mov_m * (1 - exp_w)
        elo[w] = rw + delta
        elo[l] = rl - delta
    return elo


def build_team_stats(reg_df, tourney_df):
    """Advanced stats per (Season, TeamID)."""
    eps = 1e-6
    tg  = explode_games(pd.concat([reg_df, tourney_df], ignore_index=True))
    if len(tg) == 0:
        return pd.DataFrame()

    agg = tg.groupby(["Season","TeamID"]).agg(
        GP=("Win","count"), Wins=("Win","sum"),
        PtsFor=("PtsFor","mean"), PtsAgainst=("PtsAgainst","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OR=("OR","mean"),   DR=("DR","mean"),
        Ast=("Ast","mean"), TO=("TO","mean"),
        Stl=("Stl","mean"), Blk=("Blk","mean"), PF=("PF","mean"),
        OppFGM=("OppFGM","mean"),   OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
        OppOR=("OppOR","mean"),     OppDR=("OppDR","mean"),
        OppTO=("OppTO","mean"),
    ).reset_index()

    agg["WinPct"]         = agg["Wins"] / (agg["GP"] + eps)
    agg["FGPct"]          = agg["FGM"]  / (agg["FGA"]  + eps)
    agg["FG3Pct"]         = agg["FGM3"] / (agg["FGA3"] + eps)
    agg["FTPct"]          = agg["FTM"]  / (agg["FTA"]  + eps)
    agg["OppFGPct"]       = agg["OppFGM"]  / (agg["OppFGA"]  + eps)
    agg["eFGPct"]         = (agg["FGM"]    + 0.5*agg["FGM3"])    / (agg["FGA"]    + eps)
    agg["OppEFGPct"]      = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / (agg["OppFGA"] + eps)
    poss                  = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    agg["TOVRate"]        = agg["TO"]          / (poss + eps)
    agg["ORBRate"]        = agg["OR"]          / (agg["OR"]  + agg["OppDR"] + eps)
    agg["FTRate"]         = agg["FTM"]         / (agg["FGA"] + eps)
    agg["DRBRate"]        = agg["DR"]          / (agg["DR"]  + agg["OppOR"] + eps)
    agg["Pace"]           = poss               / (agg["GP"]  + eps)
    agg["NetRtg"]         = agg["PtsFor"]      - agg["PtsAgainst"]
    agg["OffRtg"]         = agg["PtsFor"]      / (poss + eps) * 100
    agg["DefRtg"]         = agg["PtsAgainst"]  / (poss + eps) * 100
    agg["NetRtg2"]        = agg["OffRtg"]      - agg["DefRtg"]
    agg["AstTO"]          = agg["Ast"]         / (agg["TO"]     + eps)
    agg["StlBlk"]         = (agg["Stl"] + agg["Blk"]) / (agg["OppFGA"] + eps)
    agg["ThreePtRate"]    = agg["FGA3"]        / (agg["FGA"]    + eps)
    agg["OppThreePtRate"] = agg["OppFGA3"]     / (agg["OppFGA"] + eps)
    agg["TOMargin"]       = agg["OppTO"]       - agg["TO"]
    agg["RebMargin"]      = agg["OR"]          - agg["OppOR"]
    return agg


def build_recent_form(reg_df, n=10):
    """Last n regular season games per team."""
    tg  = explode_games(reg_df).sort_values(["Season","TeamID","DayNum"])
    rec = tg.groupby(["Season","TeamID"]).tail(n)
    rf  = rec.groupby(["Season","TeamID"]).agg(
        RecentWinPct    =("Win","mean"),
        RecentPtsFor    =("PtsFor","mean"),
        RecentPtsAgainst=("PtsAgainst","mean"),
    ).reset_index()
    rf["RecentNetPts"] = rf["RecentPtsFor"] - rf["RecentPtsAgainst"]
    return rf


def build_massey(seasons, day_cutoff=133):
    """Massey ordinals averaged across all systems."""
    fp = DATA_DIR / "MMasseyOrdinals.csv"
    if not fp.exists():
        print("  ℹ️  MMasseyOrdinals.csv not found — skipping")
        return None
    df = pd.read_csv(fp)
    df = df[(df["Season"].isin(seasons)) & (df["RankingDayNum"] <= day_cutoff)]
    if len(df) == 0:
        return None
    result = (df.groupby(["Season","TeamID"])["OrdinalRank"]
                .mean().reset_index()
                .rename(columns={"OrdinalRank": "AvgRank"}))
    for sys in ["SAG","POM","WOL","MOR","DOK","REW","NET"]:
        sub = df[df["SystemName"] == sys]
        if len(sub) == 0:
            continue
        s = (sub.groupby(["Season","TeamID"])["OrdinalRank"]
                .mean().reset_index()
                .rename(columns={"OrdinalRank": f"Rank_{sys}"}))
        result = result.merge(s, on=["Season","TeamID"], how="left")
    return result



# ─────────────────────────────────────────────────────────────
# NEW: CONFERENCE TOURNAMENT FEATURES
# ─────────────────────────────────────────────────────────────

def build_conf_tourney_features(conf_games_df, reg_detailed_df, seasons):
    """
    Build per-(Season, TeamID) conference tournament features.
    Conf tourney games (Day 119-132) are also in MRegularSeasonDetailedResults
    so we can get actual scores by joining on Season+DayNum+WTeamID+LTeamID.

    Features per team:
      ConfTGames       - games played (how deep they went)
      ConfTWins        - wins in conf tourney
      ConfTWinPct      - win % in conf tourney
      ConfChamp        - 1 if won their conference tournament
      ConfTAvgMargin   - avg point differential
      ConfTAvgScore    - avg points scored
      ConfTAvgAllowed  - avg points allowed
    """
    conf = conf_games_df[conf_games_df["Season"].isin(seasons)].copy()
    reg  = reg_detailed_df[reg_detailed_df["Season"].isin(seasons)].copy()

    if len(conf) == 0:
        cols = ["Season","TeamID","ConfTGames","ConfTWins","ConfTWinPct",
                "ConfChamp","ConfTAvgMargin","ConfTAvgScore","ConfTAvgAllowed"]
        return pd.DataFrame(columns=cols)

    # Get scores from detailed results — conf games are in there
    conf_scored = conf.merge(
        reg[["Season","DayNum","WTeamID","LTeamID","WScore","LScore"]],
        on=["Season","DayNum","WTeamID","LTeamID"],
        how="left"
    )

    # Conference champion = won on the final day of their conference tournament
    last_days = (conf.groupby(["Season","ConfAbbrev"])["DayNum"]
                     .max().reset_index().rename(columns={"DayNum":"LastDay"}))
    champ_games = conf.merge(last_days,
                             left_on=["Season","ConfAbbrev","DayNum"],
                             right_on=["Season","ConfAbbrev","LastDay"])
    conf_champs = set(zip(champ_games["Season"], champ_games["WTeamID"]))

    # Winner perspective
    w = conf_scored[["Season","WTeamID","WScore","LScore"]].copy()
    w["TeamID"] = w["WTeamID"]; w["Win"] = 1
    w["PtsFor"] = w["WScore"];  w["PtsAgainst"] = w["LScore"]

    # Loser perspective
    l = conf_scored[["Season","LTeamID","WScore","LScore"]].copy()
    l["TeamID"] = l["LTeamID"]; l["Win"] = 0
    l["PtsFor"] = l["LScore"];  l["PtsAgainst"] = l["WScore"]

    both = pd.concat([
        w[["Season","TeamID","Win","PtsFor","PtsAgainst"]],
        l[["Season","TeamID","Win","PtsFor","PtsAgainst"]]
    ], ignore_index=True)

    agg = both.groupby(["Season","TeamID"]).agg(
        ConfTGames      =("Win","count"),
        ConfTWins       =("Win","sum"),
        ConfTAvgScore   =("PtsFor","mean"),
        ConfTAvgAllowed =("PtsAgainst","mean"),
    ).reset_index()

    eps = 1e-6
    agg["ConfTWinPct"]    = agg["ConfTWins"] / (agg["ConfTGames"] + eps)
    agg["ConfTAvgMargin"] = agg["ConfTAvgScore"] - agg["ConfTAvgAllowed"]
    agg["ConfChamp"]      = agg.apply(
        lambda r: 1 if (r["Season"], r["TeamID"]) in conf_champs else 0, axis=1)

    return agg


def attach(df, ref, side, team_col, season_join=True):
    """
    Merge a reference table onto df for one side of a matchup.
    side      : 'A' or 'B'
    team_col  : column in df and ref that holds the team ID
    season_join: whether to also join on Season
    """
    ref    = ref.copy()
    f_cols = [c for c in ref.columns if c not in [team_col, "Season"]]
    ref    = ref.rename(columns={c: f"{side}_{c}" for c in f_cols})
    keys   = ([team_col, "Season"] if season_join and "Season" in ref.columns
              else [team_col])
    return df.merge(ref, on=keys, how="left")


def make_matchup_features(games_df, team_stats, recent, seed_feat,
                           massey_feat, conf_feat, elo_dict):
    """Build training matchup matrix. TeamA = lower ID, Label = 1 if TeamA won."""
    df          = games_df.copy()
    df["TeamA"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["TeamB"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Label"] = (df["WTeamID"] == df["TeamA"]).astype(int)

    rf = recent[["Season","TeamID","RecentWinPct","RecentNetPts",
                 "RecentPtsFor","RecentPtsAgainst"]].copy()
    sd = seed_feat[["Season","TeamID","SeedNum"]].copy()

    # Rename TeamID to TeamA / TeamB before attaching
    df = attach(df, team_stats.rename(columns={"TeamID":"TeamA"}), "A", "TeamA")
    df = attach(df, team_stats.rename(columns={"TeamID":"TeamB"}), "B", "TeamB")
    df = attach(df, rf.rename(columns={"TeamID":"TeamA"}),         "A", "TeamA")
    df = attach(df, rf.rename(columns={"TeamID":"TeamB"}),         "B", "TeamB")
    df = attach(df, sd.rename(columns={"TeamID":"TeamA"}),         "A", "TeamA")
    df = attach(df, sd.rename(columns={"TeamID":"TeamB"}),         "B", "TeamB")

    if massey_feat is not None:
        mc = [c for c in massey_feat.columns if c not in ["Season","TeamID"]]
        mf = massey_feat[["Season","TeamID"] + mc].copy()
        df = attach(df, mf.rename(columns={"TeamID":"TeamA"}), "A", "TeamA")
        df = attach(df, mf.rename(columns={"TeamID":"TeamB"}), "B", "TeamB")

    # Conference tournament features
    if conf_feat is not None and len(conf_feat) > 0:
        cf_cols = [c for c in conf_feat.columns if c not in ["Season","TeamID"]]
        cf = conf_feat[["Season","TeamID"] + cf_cols].copy()
        df = attach(df, cf.rename(columns={"TeamID":"TeamA"}), "A", "TeamA")
        df = attach(df, cf.rename(columns={"TeamID":"TeamB"}), "B", "TeamB")

    # Prior end-of-season ELO
    df["A_ELO"] = df.apply(
        lambda r: elo_dict.get((int(r["Season"])-1, int(r["TeamA"])), ELO_BASE), axis=1)
    df["B_ELO"] = df.apply(
        lambda r: elo_dict.get((int(r["Season"])-1, int(r["TeamB"])), ELO_BASE), axis=1)

    feature_cols = []
    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]
            feature_cols.extend([f"diff_{base}", ca, cb])

    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    feature_cols      = list(dict.fromkeys(feature_cols + ["ELO_WinProb"]))
    return df, feature_cols


def build_pred_features_2026(pairs_df, stats_2026, recent_2026,
                              seed_2026, massey_2026, conf_2026,
                              elo_2026_final, feature_cols):
    """
    Build 2026 prediction features.
    All tables are single-season (Season col dropped before merging).
    """
    df = pairs_df[["TeamA","TeamB"]].copy().reset_index(drop=True)

    def drop_season(t):
        return t.drop(columns=["Season"], errors="ignore").copy()

    s26  = drop_season(stats_2026)
    rf26 = drop_season(recent_2026)[["TeamID","RecentWinPct","RecentNetPts",
                                      "RecentPtsFor","RecentPtsAgainst"]]
    sd26 = drop_season(seed_2026)[["TeamID","SeedNum"]]

    df = attach(df, s26.rename(columns={"TeamID":"TeamA"}),  "A", "TeamA",  season_join=False)
    df = attach(df, s26.rename(columns={"TeamID":"TeamB"}),  "B", "TeamB",  season_join=False)
    df = attach(df, rf26.rename(columns={"TeamID":"TeamA"}), "A", "TeamA",  season_join=False)
    df = attach(df, rf26.rename(columns={"TeamID":"TeamB"}), "B", "TeamB",  season_join=False)
    df = attach(df, sd26.rename(columns={"TeamID":"TeamA"}), "A", "TeamA",  season_join=False)
    df = attach(df, sd26.rename(columns={"TeamID":"TeamB"}), "B", "TeamB",  season_join=False)

    if massey_2026 is not None and len(massey_2026) > 0:
        m26 = drop_season(massey_2026)
        mc  = [c for c in m26.columns if c != "TeamID"]
        df  = attach(df, m26[["TeamID"]+mc].rename(columns={"TeamID":"TeamA"}),
                     "A", "TeamA", season_join=False)
        df  = attach(df, m26[["TeamID"]+mc].rename(columns={"TeamID":"TeamB"}),
                     "B", "TeamB", season_join=False)

    # Conference tournament features 2026
    if conf_2026 is not None and len(conf_2026) > 0:
        cf26 = conf_2026.drop(columns=["Season"], errors="ignore").copy()
        cf_cols = [c for c in cf26.columns if c != "TeamID"]
        df = attach(df, cf26[["TeamID"]+cf_cols].rename(columns={"TeamID":"TeamA"}),
                    "A", "TeamA", season_join=False)
        df = attach(df, cf26[["TeamID"]+cf_cols].rename(columns={"TeamID":"TeamB"}),
                    "B", "TeamB", season_join=False)

    # 2026 ELO
    df["A_ELO"] = df["TeamA"].apply(lambda t: elo_2026_final.get(int(t), ELO_BASE))
    df["B_ELO"] = df["TeamB"].apply(lambda t: elo_2026_final.get(int(t), ELO_BASE))

    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]

    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))

    X = df.reindex(columns=feature_cols)
    n_nan_cols = (X.isna().sum() > 0).sum()
    if n_nan_cols > 0:
        print(f"    ℹ️  {n_nan_cols} columns had NaNs (filled 0) — normal for unseeded teams")
    return X.fillna(0)


# ─────────────────────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────────────────────
def run_pipeline(reg_full, tourney_full, seeds_full, conf_games_full,
                 use_massey, lgbm_params, xgb_params, pred_pairs, label):

    print(f"\n{'='*65}")
    print(f"  {label}")
    print(f"{'='*65}")

    reg_train     = reg_full[reg_full["Season"].isin(TRAIN_SEASONS)].copy()
    tourney_train = tourney_full[tourney_full["Season"].isin(TRAIN_SEASONS)].copy()
    reg_2026      = reg_full[reg_full["Season"] == PRED_SEASON].copy()

    print(f"  Train reg    : {len(reg_train):,} games "
          f"({reg_train.Season.min()}-{reg_train.Season.max()})")
    print(f"  Train tourney: {len(tourney_train):,} games")
    print(f"  2026 reg     : {len(reg_2026):,} games <- prediction basis")

    # ELO
    print(f"\n  [ELO] Training seasons 2010-2025 ...")
    elo_dict, elo_end_2025 = compute_elo_seasons(reg_train, tourney_train, TRAIN_SEASONS)
    print(f"        {len(elo_dict):,} (season,team) entries")

    print(f"  [ELO] Extending through 2026 regular season ...")
    elo_2026 = compute_elo_2026(reg_2026, elo_end_2025)
    print(f"        {len(elo_2026):,} teams rated for 2026")

    # Training stats
    print(f"\n  [Stats] Building 2010-2025 training stats ...")
    team_stats  = build_team_stats(reg_train, tourney_train)
    recent      = build_recent_form(reg_train)
    seed_feat   = seeds_full[seeds_full["Season"].isin(TRAIN_SEASONS)].copy()
    seed_feat["SeedNum"] = seed_feat["Seed"].apply(parse_seed_num)
    seed_feat   = seed_feat[["Season","TeamID","SeedNum"]]
    massey_train = build_massey(TRAIN_SEASONS) if use_massey else None
    print(f"        {len(team_stats):,} team-season rows")
    print(f"        Massey: {'yes' if massey_train is not None else 'N/A'}")

    # Conference tourney training features
    print(f"  [ConfT] Building conference tourney features 2010-2025 ...")
    conf_train = build_conf_tourney_features(conf_games_full, reg_train, TRAIN_SEASONS)
    print(f"         {len(conf_train):,} team-season rows | 7 features")

    # Training matchup matrix
    print(f"\n  [Features] Building training matchup matrix ...")
    all_train = pd.concat([reg_train, tourney_train], ignore_index=True)
    train_df, feature_cols = make_matchup_features(
        all_train, team_stats, recent, seed_feat, massey_train, conf_train, elo_dict)
    X_tr = train_df[feature_cols].fillna(0)
    y_tr = train_df["Label"]
    print(f"        {len(X_tr):,} rows x {len(feature_cols)} features")

    # Train LightGBM
    print(f"\n  [Train] LightGBM ...")
    lgbm_m = lgb.LGBMClassifier(**lgbm_params)
    lgbm_m.fit(X_tr, y_tr)
    print(f"         done")

    # Train XGBoost (needs val split for early stopping)
    split  = int(len(X_tr) * 0.9)
    X_trx, y_trx   = X_tr.iloc[:split],  y_tr.iloc[:split]
    X_valx, y_valx = X_tr.iloc[split:],  y_tr.iloc[split:]

    print(f"  [Train] XGBoost ...")
    xgb_m = xgb.XGBClassifier(**xgb_params)
    xgb_m.fit(X_trx, y_trx, eval_set=[(X_valx, y_valx)], verbose=False)
    print(f"         done")

    # Tune blend weight
    p_lv = lgbm_m.predict_proba(X_valx)[:,1]
    p_xv = xgb_m.predict_proba(X_valx)[:,1]
    best_w, best_ll = 0.5, 99.0
    for w in np.arange(0.05, 0.96, 0.05):
        ll = log_loss(y_valx, w*p_lv + (1-w)*p_xv)
        if ll < best_ll:
            best_ll, best_w = ll, w
    print(f"  [Blend] LGB:{best_w:.2f}  XGB:{1-best_w:.2f}  "
          f"val_logloss={best_ll:.5f}")

    # 2026 prediction features  <-- THE KEY FIX
    print(f"\n  [2026] Building 2026 prediction features ...")
    empty_df    = pd.DataFrame(columns=reg_2026.columns)
    stats_2026  = build_team_stats(reg_2026, empty_df)
    stats_2026  = stats_2026[stats_2026["Season"] == PRED_SEASON].copy()
    recent_2026 = build_recent_form(reg_2026)
    recent_2026 = recent_2026[recent_2026["Season"] == PRED_SEASON].copy()
    seed_2026   = seeds_full[seeds_full["Season"] == PRED_SEASON].copy()
    seed_2026["SeedNum"] = seed_2026["Seed"].apply(parse_seed_num)
    seed_2026   = seed_2026[["Season","TeamID","SeedNum"]]
    massey_2026 = None
    if use_massey:
        massey_2026 = build_massey([PRED_SEASON])
        if massey_2026 is not None:
            massey_2026 = massey_2026[massey_2026["Season"] == PRED_SEASON].copy()

    # Conference tourney 2026
    conf_2026_feat = build_conf_tourney_features(conf_games_full, reg_2026, [PRED_SEASON])
    conf_2026_feat = conf_2026_feat[conf_2026_feat["Season"] == PRED_SEASON].copy()

    print(f"        stats  : {len(stats_2026):,} teams")
    print(f"        recent : {len(recent_2026):,} teams")
    print(f"        seeds  : {len(seed_2026):,} teams")
    print(f"        conf_t : {len(conf_2026_feat):,} teams | {int(conf_2026_feat['ConfChamp'].sum())} conf champs")
    print(f"        massey : {len(massey_2026) if massey_2026 is not None else 'N/A'}")

    X_pred = build_pred_features_2026(
        pred_pairs, stats_2026, recent_2026,
        seed_2026, massey_2026, conf_2026_feat, elo_2026, feature_cols)
    print(f"        {len(X_pred):,} prediction rows")

    # Predict
    print(f"\n  [Predict] Running ensemble ...")
    p_lgbm = lgbm_m.predict_proba(X_pred)[:,1]
    p_xgb  = xgb_m.predict_proba(X_pred)[:,1]
    p_ens  = np.clip(best_w*p_lgbm + (1-best_w)*p_xgb, 0.025, 0.975)
    print(f"         mean={p_ens.mean():.4f}  "
          f"min={p_ens.min():.4f}  max={p_ens.max():.4f}")
    return p_ens


# ─────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  LOADING DATA")
print("="*65)

stage2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")
parts           = stage2["ID"].str.split("_", expand=True)
stage2["Season"] = parts[0].astype(int)
stage2["TeamA"]  = parts[1].astype(int)
stage2["TeamB"]  = parts[2].astype(int)

men_mask    = stage2["TeamA"] < 3000
women_mask  = stage2["TeamA"] >= 3000
men_pairs   = stage2[men_mask][["TeamA","TeamB"]].reset_index(drop=True)
women_pairs = stage2[women_mask][["TeamA","TeamB"]].reset_index(drop=True)

print(f"  Submission total: {len(stage2):,}")
print(f"  Men    (1xxx)   : {len(men_pairs):,}")
print(f"  Women  (3xxx)   : {len(women_pairs):,}")

print("\n  Men's files ...")
m_reg_full     = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney_full = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
m_seeds_full      = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_conf_games_full = pd.read_csv(DATA_DIR / "MConferenceTourneyGames.csv")
print(f"  M reg {m_reg_full.Season.min()}-{m_reg_full.Season.max()} | "
      f"2026 games: {len(m_reg_full[m_reg_full.Season==2026]):,}")
print(f"  M seeds 2026: {len(m_seeds_full[m_seeds_full.Season==2026])} teams")

print("\n  Women's files ...")
w_reg_full     = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney_full = pd.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")
w_seeds_full      = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
w_conf_games_full = pd.read_csv(DATA_DIR / "WConferenceTourneyGames.csv")
print(f"  W reg {w_reg_full.Season.min()}-{w_reg_full.Season.max()} | "
      f"2026 games: {len(w_reg_full[w_reg_full.Season==2026]):,}")
print(f"  W seeds 2026: {len(w_seeds_full[w_seeds_full.Season==2026])} teams")

# ─────────────────────────────────────────────────────────────
# RUN PIPELINES
# ─────────────────────────────────────────────────────────────
men_preds = run_pipeline(
    m_reg_full, m_tourney_full, m_seeds_full,
    conf_games_full=m_conf_games_full,
    use_massey=True,
    lgbm_params=MEN_LGBM, xgb_params=MEN_XGB,
    pred_pairs=men_pairs,
    label="MENS PIPELINE"
)

women_preds = run_pipeline(
    w_reg_full, w_tourney_full, w_seeds_full,
    conf_games_full=w_conf_games_full,
    use_massey=False,
    lgbm_params=WOMEN_LGBM, xgb_params=WOMEN_XGB,
    pred_pairs=women_pairs,
    label="WOMENS PIPELINE"
)

# ─────────────────────────────────────────────────────────────
# ASSEMBLE + SAVE
# ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  ASSEMBLING SUBMISSION")
print("="*65)

men_out         = stage2[men_mask][["ID"]].copy().reset_index(drop=True)
men_out["Pred"] = men_preds.round(6)

women_out         = stage2[women_mask][["ID"]].copy().reset_index(drop=True)
women_out["Pred"] = women_preds.round(6)

submission = (stage2[["ID"]]
              .merge(pd.concat([men_out, women_out], ignore_index=True),
                     on="ID", how="left"))
submission["Pred"] = submission["Pred"].clip(0.025, 0.975).round(6)

# Lock confirmed First Four results
# ID format: 2026_loTeamID_hiTeamID | Pred = P(loTeamID wins)
known_results = {
    "2026_1224_1420": 1.0,   # Howard beat UMBC       86-83
    "2026_1301_1400": 0.0,   # Texas  beat NC State   68-66  (1301=NC State lost)
    "2026_1250_1341": 0.0,   # Prairie View beat Lehigh 67-55 (1250=Lehigh lost)
    "2026_1275_1374": 1.0,   # Miami OH beat SMU       89-79  (1275=Miami OH won)
}
id_idx = {row["ID"]: i for i, row in submission.iterrows()}
for mid, val in known_results.items():
    if mid in id_idx:
        submission.at[id_idx[mid], "Pred"] = val
        print(f"  Locked: {mid} -> {val}")

# Save
out_path = OUT_DIR / "shaurya_final_6.csv"
submission[["ID","Pred"]].to_csv(out_path, index=False)

# Sanity checks
assert len(submission) == len(stage2),               "Row count mismatch"
assert submission["ID"].nunique() == len(submission), "Duplicate IDs"
assert submission["Pred"].between(0,1).all(),         "Pred outside [0,1]"
assert submission["Pred"].isna().sum() == 0,          "NaN predictions"

print(f"\n{'='*65}")
print(f"  DONE")
print(f"  File    : {out_path}")
print(f"  Rows    : {len(submission):,}  (expected 132,133)")
print(f"  NaNs    : {submission['Pred'].isna().sum()}")
print(f"  Min pred: {submission['Pred'].min():.4f}")
print(f"  Max pred: {submission['Pred'].max():.4f}")
print(f"  Mean    : {submission['Pred'].mean():.4f}")
print(f"{'='*65}")
print(f"\n  shaurya_final_6.csv is ready to submit on Kaggle!")
print(submission.head(5).to_string(index=False))

Data dir  : /Users/shaurya/Downloads
Output dir: /Users/shaurya/Downloads
CPU cores : 8

  LOADING DATA
  Submission total: 132,133
  Men    (1xxx)   : 66,430
  Women  (3xxx)   : 65,703

  Men's files ...
  M reg 2003-2026 | 2026 games: 5,647
  M seeds 2026: 68 teams

  Women's files ...
  W reg 2010-2026 | 2026 games: 5,479
  W seeds 2026: 68 teams

  MENS PIPELINE
  Train reg    : 84,808 games (2010-2025)
  Train tourney: 1,001 games
  2026 reg     : 5,647 games <- prediction basis

  [ELO] Training seasons 2010-2025 ...
        5,690 (season,team) entries
  [ELO] Extending through 2026 regular season ...
        370 teams rated for 2026

  [Stats] Building 2010-2025 training stats ...
        5,639 team-season rows
        Massey: yes
  [ConfT] Building conference tourney features 2010-2025 ...
         5,006 team-season rows | 7 features

  [Features] Building training matchup matrix ...
        85,809 rows x 202 features

  [Train] LightGBM ...
         done
  [Train] XGBoost ...
